# Оптимизация сети Джексона с ограничением бюджета  
**Студент:** *Перминов Н.А.*

В работе рассматривается открытая сеть серверов (Jackson‑сеть) со следующими характеристиками:  
- **Модели очередей** – FIFO или Processor Sharing (PS).  
- **Скорость обслуживания** `µ` и внешняя интенсивность поступления `λ` известны для узла `chain1`.  
- Межузловые переходы заданы вероятностями.  

Наша цель – *оптимизировать взвешенный параметр*  

$$
\Phi = \frac{\text{пропускная способность (throughput)}}{\text{средняя задержка}}
$$

путём добавления 1–2 новых серверов, при этом суммарные затраты на новые узлы не должны превышать  
10% от исходной стоимости сети.  

Для анализа используется **теорема Джексона**: для открытой сети с независимыми M/M/1‑очередями
статистическое распределение в состоянии системы имеет произведённую форму, а пропускная способность
не зависит от дисциплины обслуживания (FIFO↔PS).  
Это позволяет оценивать влияние изменения числа серверов без детального моделирования.

Ниже приведён Jupyter‑Notebook, реализующий все расчёты и выводы.  

In [1]:
import yaml
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

## Загрузка конфигурации сети из YAML

In [2]:
yaml_str = """
servers:
  - id: chain1
    mu: 7.159457094969063
    queue_type: FIFO
    lambda: 3.579728547484531
  - id: chain2
    mu: 2.367157068548567
    queue_type: PS
  - id: chain3
    mu: 6.084057042335131
    queue_type: FIFO
  - id: chain4
    mu: 1.197904448724397
    queue_type: PS
transitions:
  - from: chain1
    to: exit
    probability: 0.7998614202260583
  - from: chain1
    to: chain3
    probability: 0.1579390742381085
  - from: chain1
    to: chain2
    probability: 0.04219950553583315
  - from: chain2
    to: exit
    probability: 0.4890477719917962
  - from: chain2
    to: chain1
    probability: 0.2312069757232702
  - from: chain2
    to: chain4
    probability: 0.2797452522849335
  - from: chain3
    to: exit
    probability: 0.7830293746341181
  - from: chain3
    to: chain1
    probability: 0.1126989522087693
  - from: chain3
    to: chain2
    probability: 0.1042716731571126
  - from: chain4
    to: exit
    probability: 0.6284635396431909
  - from: chain4
    to: chain1
    probability: 0.3205993083362966
  - from: chain4
    to: chain2
    probability: 0.05093715202051252
"""
config = yaml.safe_load(yaml_str)

In [3]:
# config

## Парсинг результатов симуляции

In [4]:
sim_text = """
Simulation results (max time: 1000):
Server 'chain1'
  Type: FIFO
  Cost: 26628.9
  Jobs: 3642
  Avg queue time: 0.137684
  Avg service time: 0.141404
  Utilization: 0.514992
Server 'chain2'
  Type: PS
  Cost: 3801.72
  Jobs: 217
  Avg queue time: 0.0280452
  Avg service time: 0.431911
  Utilization: 0.0937246
Server 'chain3'
  Type: FIFO
  Cost: 19507.9
  Jobs: 560
  Avg queue time: 0.0110175
  Avg service time: 0.161758
  Utilization: 0.0905843
Server 'chain4'
  Type: PS
  Cost: 1717.49
  Jobs: 61
  Avg queue time: 0.0936046
  Avg service time: 1.06988
  Utilization: 0.0652628

Total system cost: 51656
Total tasks: 3496
Mean time: 0.360608
"""


def parse_simulation(text):
    server_blocks = text.strip().split("Server")
    data = {}
    for block in server_blocks[1:]:
        # id
        m_id = re.search(r"'([^']+)'", block)
        if not m_id:
            continue
        srv_id = m_id.group(1)

        # key-value pairs
        kvs = {}
        for line in block.splitlines():
            m = re.match(r"\s*(.+?):\s*(.*)", line)
            if m:
                k, v = m.group(1).strip(), m.group(2)
                kvs[k] = v
        # convert to proper types
        parsed = {}
        for k, v in kvs.items():
            if k.strip() == 'Type':
                parsed['Type'] = v.strip()
            elif k.strip() == 'Cost':
                parsed['Cost'] = float(v)
            elif k.strip() == 'Jobs':
                parsed['Jobs'] = int(v)
            elif k.strip() == 'Avg queue time':
                parsed['Avg_queue_time'] = float(v)
            elif k.strip() == 'Avg service time':
                parsed['Avg_service_time'] = float(v)
            elif k.strip() == 'Utilization':
                parsed['Utilization'] = float(v)
        data[srv_id] = parsed
    # parse global metrics
    total_cost = re.search(r"Total system cost:\s*([-\d\.]+)", text).group(1)
    total_tasks = re.search(r"Total tasks:\s*(\d+)", text).group(1)
    mean_time = re.search(r"Mean time:\s*([-\d\.]+)", text).group(1)

    return data, float(total_cost), int(total_tasks), float(mean_time)


sim_data, total_cost, total_tasks, mean_sojourn = parse_simulation(sim_text)

print(f"Total system cost:  {total_cost:.3f}")
print(f"Total tasks:        {total_tasks:.3f}")
print(f"Mean time:          {mean_sojourn:.3f}")

Total system cost:  51656.000
Total tasks:        3496.000
Mean time:          0.361


In [5]:
pd.DataFrame.from_dict(sim_data, orient='index')

,Type,Cost,Jobs,Avg_queue_time,Avg_service_time,Utilization
chain1,FIFO,26628.90,3642,0.137684,0.141404,0.514992
chain2,PS,3801.72,217,0.028045,0.431911,0.093725
chain3,FIFO,19507.90,560,0.011017,0.161758,0.090584
chain4,PS,1717.49,61,0.093605,1.069880,0.065263


In [6]:
for srv in config['servers']:
    sid = srv['id']
    data = sim_data.get(sid)
    if data:
        srv['Cost'] = data['Cost']
        srv['Jobs'] = data['Jobs']
        srv['Avg_queue_time'] = data['Avg_queue_time']
        srv['Avg_service_time'] = data['Avg_service_time']
        srv['Utilization'] = data['Utilization']
# config

## Основные метрики сети

### Пропускная способность и средняя задержка

*Пропускная способность* – это количество задач, обработанных за единицу времени  
(средний throughput). Поскольку симуляция проводилась в течение `max time = 1000` секунд,  

$$
\lambda_{\text{net}}=\frac{\text{Total tasks}}{\text{Sim. time}}
$$

*Средняя задержка* – это уже заданный показатель mean sojourn time.

In [7]:
sim_time = 1000  # как указано в выводе симуляции
throughput_net = total_tasks / sim_time
mean_delay_net = mean_sojourn

print(f"Throughput (tasks/s):  {throughput_net:.3f}")
print(f"Mean delay (s):        {mean_delay_net:.5f}")

# Взвешенный параметр Φ
phi_net = throughput_net / mean_delay_net
print(f"Φ (throughput/delay):  {phi_net:.3f}")

Throughput (tasks/s):  3.496
Mean delay (s):        0.36061
Φ (throughput/delay):  9.695


### Таблица по серверам

In [8]:
df_srv = pd.DataFrame.from_dict(sim_data, orient='index')
df_srv['mu'] = df_srv.index.map(lambda x: next((s['mu'] for s in config['servers'] if s['id'] == x), np.nan))
df_srv['lambda_ext'] = None
# только chain1 имеет внешнюю интенсивность λ в конфиге
for srv in config['servers']:
    if 'lambda' in srv:
        df_srv.at[srv['id'], 'lambda_ext'] = srv['lambda']

df_srv[['Type', 'Cost', 'Jobs', 'Avg_queue_time', 'Avg_service_time', 'Utilization', 'mu', 'lambda_ext']]

,Type,Cost,Jobs,Avg_queue_time,Avg_service_time,Utilization,mu,lambda_ext
chain1,FIFO,26628.90,3642,0.137684,0.141404,0.514992,7.159457,3.579729
chain2,PS,3801.72,217,0.028045,0.431911,0.093725,2.367157,None
chain3,FIFO,19507.90,560,0.011017,0.161758,0.090584,6.084057,None
chain4,PS,1717.49,61,0.093605,1.069880,0.065263,1.197904,None


## Расчёт потоков и загрузки узлов

В открытой сети Джексона интенсивность задач в каждом узле удовлетворяет уравнению  
$$
\lambda_i = \lambda_{\text{ext},i} + \sum_j \lambda_j\,p_{j,i},
$$
где $p_{j,i}$ – вероятность перехода из узла $j$ в узел $i$.  
Мы решаем эту систему линейных уравнений и получаем $\lambda$, а затем
вычисляем коэффициент загрузки  
$$
\rho_i=\frac{\lambda_i}{\mu_i}.
$$
Самый загруженный узел – кандидат для добавления новой мощности.

## Визит‑рекуррентные коэффициенты и средняя задержка

Визит‑распределение $v_i$ описывает, сколько раз в среднем задача посещает
узел $i$. Оно удовлетворяет той же системе уравнений, но с единичным
внешним потоком на узле‑источнике (`chain1`):
$$
v = e + A^T v \quad \Longrightarrow \quad (I-A)v=e.
$$
Средняя задержка задачи в сети равна  
$$
W_{\text{net}}=\sum_i v_i\,w_i,\qquad
w_i=\frac{1}{\mu_i-\lambda_i}
$$
для узлов M/M/1 (FIFO и PS дают одинаковый результат).  Пропускная способность сети –
это интенсивность внешнего потока:
$$
\Phi = \frac{\lambda_{\text{ext}}}{W_{\text{net}}}
$$


In [24]:
# --- Визит‑распределения ---
e_vec = np.zeros(n_nodes)
source_node = 'chain1'
e_vec[node_index[source_node]] = 1.0
v = np.linalg.solve(I_minus_A, e_vec)

# --- Средняя задержка (среднее время ожидания+обслуживания) ---
W = np.clip(1/(mu_vals - lam), 0, None)  # w_i
mean_delay_theoretical = np.dot(v, W)
throughput = lambda_ext_vec[node_index[source_node]]   # λ_ext
phi_original = throughput / mean_delay_theoretical

print(f"\nВизит‑распределения v_i:")
for nid, vi in zip(nodes, v):
    print(f"  {nid:>6}: {vi:.4f}")
print("\nСредняя задержка сети (теория):", f"{mean_delay_theoretical:.5f} s")
print("Пропускная способность λ_ext:", f"{throughput:.5f} задач/с")
print("Φ = throughput / delay:", f"{phi_original:.3f}\n")


Визит‑распределения v_i:
  chain1: 1.0383
  chain2: 0.3380
  chain3: 0.1523
  chain4: 0.3501

Средняя задержка сети (теория): 0.62119 s
Пропускная способность λ_ext: 3.57973 задач/с
Φ = throughput / delay: 5.763



## Стоимость новых серверов и ограничение бюджета

Стоимость каждого узла известна из результатов симуляции:
$$
C_i=\text{Cost}_i.
$$
Ниже вычисляем коэффициент «стоимость/скорость обслуживания» $c_{\mu,i}=C_i/\mu_i$. Это позволяет сравнить, какой тип узла экономичнее.  

Бюджет для расширения сети равен 10% от исходной стоимости:
$$
B=0.1 \times C_{\text{total}}.
$$  
Ниже показываем бюджет и самые экономичные варианты добавления новых серверов.``` 

## 4. Выбор параметров нового сервера (определяем переменные)

В этой ячейке задаём все параметры, которые будут использоваться при расчёте добавления одного нового узла:
- `target_node` – id узла, к которому будет подключён новый сервер  
  (выбирается автоматически как самый загруженный узел, но можно переопределить);
- `new_mu` – скорость обслуживания нового сервера;
- `cost_per_mu` – стоимость единицы скорости для выбранного типа узла
  (рассчитывается из исходных данных).  
После того как эти переменные заданы, остальные ячейки автоматически
используют их для вычислений.

## 5. Расчёт влияния добавления одного сервера

В этой ячейке используются переменные из предыдущего блока (`target_node`, `new_mu`, `cost_new`) для:
- обновления скорости обслуживания выбранного узла;
- пересчёта среднего времени ожидания в этом узле;
- вычисления новой средней задержки сети и улучшения показателя Φ.